# Time Shifting & Rolling Windows

In real-world chronological datasets (such as daily revenues or sensor readings), a value's meaning is highly tied to its context over time .

To analyze historical changes, we use:
*   **`.shift()`**: Moves data rows up or down, allowing you to align past values next to current ones (e.g., comparing today's revenue directly with yesterday's).
*   **`.rolling()`**: Sets up moving/rolling calculation windows (e.g., a 7-day moving average), which smooths out short-term fluctuations and reveals long-term trends.


### Simple Explanation & Real-World Analogy
*   **`.shift()`** is like looking at a calendar of daily steps and copying yesterday's count onto today's row so you can see if you walked more or less than the day before.
*   **`.rolling()`** is like looking at the weather. Instead of judging the temperature based solely on a single extreme afternoon, you take the average of the last 3 days to get a smooth, reliable trend of the climate.

### Code Examples

Let's build a timeline of daily store revenue and use both shifting and rolling calculations.


In [2]:
import pandas as pd

sales_timeline = pd.DataFrame({
    'Revenue': [120, 150, 110, 200, 250, 180, 210]
}, index=pd.to_datetime([
    '2026-08-17', '2026-08-18', '2026-08-19', '2026-08-20',
    '2026-08-21', '2026-08-22', '2026-08-23'
]))

# A) Shifting data by 1 row to get "Yesterday's Revenue"
sales_timeline['Yesterday_Revenue'] = sales_timeline['Revenue'].shift(1)

# Calculating the daily change in revenue
sales_timeline['Daily_Change'] = sales_timeline['Revenue'] - sales_timeline['Yesterday_Revenue']

print("--- Shifting Example ---")
print(sales_timeline)

--- Shifting Example ---
            Revenue  Yesterday_Revenue  Daily_Change
2026-08-17      120                NaN           NaN
2026-08-18      150              120.0          30.0
2026-08-19      110              150.0         -40.0
2026-08-20      200              110.0          90.0
2026-08-21      250              200.0          50.0
2026-08-22      180              250.0         -70.0
2026-08-23      210              180.0          30.0


*Note: The first row contains `NaN` for yesterday's revenue because there is no recorded day before it to shift down.*



#### Rolling Average (Moving Windows)
Let's compute a **3-day rolling average** of the revenue. This will take the current day and the preceding 2 days, and calculate their average.

In [4]:
# Rolling window of size 3, computing the mean [47]
sales_timeline['3Day_Rolling_Avg'] = sales_timeline['Revenue'].rolling(window=3).mean()

print("--- Rolling Window Example ---")
print(sales_timeline[['Revenue', '3Day_Rolling_Avg']])

--- Rolling Window Example ---
            Revenue  3Day_Rolling_Avg
2026-08-17      120               NaN
2026-08-18      150               NaN
2026-08-19      110        126.666667
2026-08-20      200        153.333333
2026-08-21      250        186.666667
2026-08-22      180        210.000000
2026-08-23      210        213.333333


*Note: The first two rows are `NaN` because a 3-day window requires at least 3 historical values to calculate a valid mean.*


### Common Pitfalls to Avoid
1. **Shifting Mixed Groups Incorrectly**: If your DataFrame contains data for multiple stores, calling `.shift(1)` globally will accidentally shift the last row of "Store A" into the first row of "Store B". When working with groups, always use `.groupby('Store')['Revenue'].shift(1)` to keep divisions isolated!
2. **Forgetting Chronological Sorting**: Rolling window computations assume your index is arranged in correct chronological order. If your dates are randomized or out of order, `.rolling()` will calculate averages of unrelated, scrambled dates. Always run `.sort_index()` first!

#### Exercise 1 (Easy)
Given the following series of stock closing prices, shift the prices backwards by 1 day (shift -1) to align "Tomorrow's Price" next to each current row.
```python
stock_prices = pd.Series([100, 105, 102, 110])
```

In [5]:
import pandas as pd
stock_prices = pd.Series([100, 105, 102, 110])

# Shifting with a negative step shifts values upward
tomorrow_prices = stock_prices.shift(-1)
print(tomorrow_prices)

0    105.0
1    102.0
2    110.0
3      NaN
dtype: float64


#### Exercise 2 (Medium)
Given a series of daily temperature readings, calculate a **2-day rolling sum** of the temperature readings.
```python
temps = pd.Series([20, 22, 25, 24, 21])
```

In [7]:
import pandas as pd
temps = pd.Series([20, 22, 25, 24, 21])

# Setting window=2 and calculating sum [47]
rolling_sums = temps.rolling(window=2).sum()
print(rolling_sums)

0     NaN
1    42.0
2    47.0
3    49.0
4    45.0
dtype: float64
